# Writing Tools NLP — QLoRA fine-tuning (Kaggle GPU)

Fine-tune **Qwen3-4B-Instruct-2507** with QLoRA on ~4k writing-task examples.

**Before running:** Notebook settings → **Accelerator: GPU** (T4 or P100), **Internet: On**.

**Expected runtime:** ~6–8h training + ~1–2h eval (9h session limit). Mid-training eval is disabled to avoid OOM; quality metrics run via `train/eval.py` after training.

**Output:** `adapter.zip` in `/kaggle/working/` — download from the Kaggle **Output** tab when the notebook completes.

In [ ]:
# Cell 1 — GPU check + clone repo
import torch

assert torch.cuda.is_available(), "Enable GPU: Settings → Accelerator → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

!git clone -b kaggel_free https://github.com/Aditya-ice/writing-tools-NLP.git
%cd writing-tools-NLP

In [ ]:
# Cell 2 — Optional HuggingFace login (add HF_TOKEN in Kaggle Secrets)
import os

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    !huggingface-cli login --token $HF_TOKEN
    print("Logged in to HuggingFace.")
except Exception as e:
    print(f"Skipping HF login (optional): {e}")

In [ ]:
# Cell 3 — Install dependencies (mlx excluded on Linux via requirements.txt markers)
%pip install -q -r requirements.txt

In [ ]:
# Cell 4 — Prepare datasets (~4k examples, ~30s)
!python data/prepare.py

In [ ]:
# Cell 5 — Train QLoRA adapter (~6–8h on GPU for 1 epoch)
# Resume after a disconnect:
#   !python train/train.py --resume_from_checkpoint ./checkpoints/checkpoint-XXX

!python train/train.py

In [ ]:
# Cell 6 — ROUGE + BERTScore eval (base vs fine-tuned)
# Quick smoke test if nearing 9h session limit:
#   !python train/eval.py --adapter_path ./adapter --split val --max_samples 50

!python train/eval.py --adapter_path ./adapter --split val

In [ ]:
# Cell 7 — Package adapter for download (persists in /kaggle/working/)
import shutil

shutil.make_archive("/kaggle/working/adapter", "zip", "adapter")
print("Saved /kaggle/working/adapter.zip — download from the Output tab.")